In [14]:
import onnx
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.basic import get_by_name

FINN_ONNX_PATH = '/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_qonnx.onnx'

model = ModelWrapper(FINN_ONNX_PATH)

from qonnx.core.modelwrapper import ModelWrapper

model_path = FINN_ONNX_PATH   # 确认这里就是 verify_qonnx_vs_finn() 用的那个路径
print("model_path =", model_path)

model = ModelWrapper(model_path)

print("\n=== all custom-domain nodes ===")
for n in model.graph.node:
    if n.domain != "":
        print("name =", n.name, "| op_type =", n.op_type, "| domain =", n.domain)

print("\n=== nodes with 'Threshold' in op/name ===")
for n in model.graph.node:
    if "Threshold" in n.op_type or "Threshold" in n.name:
        print("name =", n.name, "| op_type =", n.op_type, "| domain =", n.domain)

print("\n=== MultiThreshold details ===")
found = False
for n in model.graph.node:
    if n.op_type == "MultiThreshold":
        found = True
        inp_shape = model.get_tensor_shape(n.input[0])
        th_name = n.input[1]
        th = model.get_initializer(th_name)
        print(
            "name =", n.name,
            "| input =", n.input[0],
            "| input_shape =", inp_shape,
            "| thresholds =", th_name,
            "| threshold_shape =", None if th is None else th.shape,
            "| domain =", n.domain,
        )

if not found:
    print("No MultiThreshold nodes found in this exact file.")

model_path = /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_qonnx.onnx

=== all custom-domain nodes ===
name = /quant_in/act_quant/export_handler/Quant | op_type = Quant | domain = onnx.brevitas
name = /backbone/network.0/conv1/conv/input_quant/export_handler/Quant | op_type = Quant | domain = onnx.brevitas
name = /backbone/network.0/conv1/conv/weight_quant/export_handler/Quant | op_type = Quant | domain = onnx.brevitas
name = /backbone/network.0/conv1/conv/bias_quant/export_handler/Quant | op_type = Quant | domain = onnx.brevitas
name = /backbone/network.0/activation1/act_quant/export_handler/Quant | op_type = Quant | domain = onnx.brevitas
name = /backbone/network.0/conv2/conv/input_quant/export_handler/Quant | op_type = Quant | domain = onnx.brevitas
name = /backbone/network.0/conv2/conv/weight_quant/export_handler/Quant | op_type = Quant | domain = onnx.brevitas
name = /backbone/network.0/conv2/conv/bias_quant/export_handler/Quant |

In [24]:
import onnx
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp
FINN_ONNX_PATH = '/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx'
model = ModelWrapper(FINN_ONNX_PATH)

print("=== MultiThreshold nodes ===")
for node in model.graph.node:
    if node.op_type == "MultiThreshold":
        mt = getCustomOp(node)
        th_name = node.input[1]
        th = model.get_initializer(th_name)

        print("-" * 80)
        print("node name   :", node.name)
        print("input tensor:", node.input[0])
        print("th tensor   :", th_name)
        print("th shape    :", None if th is None else th.shape)

        in_shape = model.get_tensor_shape(node.input[0])
        out_shape = model.get_tensor_shape(node.output[0])

        print("input shape :", in_shape)
        print("output shape:", out_shape)

=== MultiThreshold nodes ===
--------------------------------------------------------------------------------
node name   : 
input tensor: x.1367
th tensor   : 3RSo1Z
th shape    : (1, 255)
input shape : [1, 1, 140]
output shape: [1, 1, 140]
--------------------------------------------------------------------------------
node name   : 
input tensor: /backbone/Unsqueeze_output_0
th tensor   : 1d894D
th shape    : (1, 255)
input shape : [1, 1, 1, 140]
output shape: [1, 1, 1, 140]
--------------------------------------------------------------------------------
node name   : 
input tensor: /backbone/network.0/conv1/padder/Pad_output_0
th tensor   : yo5hSt
th shape    : (0, 255)
input shape : [0, 0, 0, 0]
output shape: [0, 0, 0, 0]
--------------------------------------------------------------------------------
node name   : 
input tensor: /backbone/network.0/conv1/conv/Conv_output_0
th tensor   : HqKI4p
th shape    : (16, 255)
input shape : [0, 16, 0, 0]
output shape: [0, 16, 0, 0]
-------

In [25]:
import onnx
import os
FINN_ONNX_PATH = '/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx'
from qonnx.core.modelwrapper import ModelWrapper


from qonnx.custom_op.registry import getCustomOp

from collections import Counter

def print_node_neighbors(model, target_name):
    nodes = list(model.graph.node)

    producer = {}
    consumers = {}
    for n in nodes:
        for o in n.output:
            producer[o] = n
        for i in n.input:
            consumers.setdefault(i, []).append(n)

    for n in nodes:
        if n.name != target_name:
            continue

        print("\n" + "=" * 100)
        print("TARGET:", n.name, n.op_type, n.domain)

        print("\nProducers:")
        for inp in n.input:
            p = producer.get(inp, None)
            if p is not None:
                print(" ", p.name, p.op_type, p.domain, "->", inp)

        print("\nConsumers:")
        for out in n.output:
            for c in consumers.get(out, []):
                print(" ", out, "->", c.name, c.op_type, c.domain)


finn_model = ModelWrapper(FINN_ONNX_PATH)

ops = Counter([n.op_type for n in finn_model.graph.node])
print("Loaded FINN model:", FINN_ONNX_PATH)
print("Num nodes:", len(finn_model.graph.node))
print("Num MultiThreshold:", ops["MultiThreshold"])
print("Top ops:", ops.most_common(20))

for n in finn_model.graph.node:
    if n.op_type == "MultiThreshold":
        print("FOUND MT:", n.name, "| domain =", n.domain)

print_node_neighbors(finn_model, "Quant")



Loaded FINN model: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx
Num nodes: 131
Num MultiThreshold: 19
Top ops: [('Mul', 37), ('MultiThreshold', 19), ('Add', 14), ('Reshape', 12), ('Conv', 8), ('ConstantOfShape', 6), ('Concat', 6), ('Slice', 6), ('Transpose', 6), ('Cast', 6), ('Pad', 6), ('Unsqueeze', 1), ('GlobalAveragePool', 1), ('Flatten', 1), ('MatMul', 1), ('Div', 1)]
FOUND MT:  | domain = qonnx.custom_op.general
FOUND MT:  | domain = qonnx.custom_op.general
FOUND MT:  | domain = qonnx.custom_op.general
FOUND MT:  | domain = qonnx.custom_op.general
FOUND MT:  | domain = qonnx.custom_op.general
FOUND MT:  | domain = qonnx.custom_op.general
FOUND MT:  | domain = qonnx.custom_op.general
FOUND MT:  | domain = qonnx.custom_op.general
FOUND MT:  | domain = qonnx.custom_op.general
FOUND MT:  | domain = qonnx.custom_op.general
FOUND MT:  | domain = qonnx.custom_op.general
FOUND MT:  | domain = qonnx.custom_op.general
FOUND MT:  

In [14]:
from qonnx.core.modelwrapper import ModelWrapper

model = ModelWrapper(FINN_ONNX_PATH)
nodes = list(model.graph.node)

producer = {}
consumers = {}
for n in nodes:
    for o in n.output:
        producer[o] = n
    for i in n.input:
        consumers.setdefault(i, []).append(n)

def node_desc(n):
    return f"{n.name} | {n.op_type} | domain={n.domain}"

for idx, n in enumerate(nodes):
    if n.op_type != "MultiThreshold":
        continue

    print("=" * 120)
    print("TARGET:", node_desc(n))

    print("\nProducers:")
    for inp in n.input:
        p = producer.get(inp, None)
        if p is not None:
            print("  ", node_desc(p), "->", inp)

    print("\nConsumers:")
    for out in n.output:
        for c in consumers.get(out, []):
            print("  ", out, "->", node_desc(c))

In [2]:
import onnx

m = onnx.load(CLEAN_QONNX_PATH)

for node in m.graph.node:
    if node.op_type != "If":
        continue
    print(f"\n===== {node.name} =====")
    print("inputs :", list(node.input))
    print("outputs:", list(node.output))
    for attr in node.attribute:
        if attr.name in ("then_branch", "else_branch"):
            print(f"\n  [{attr.name}]")
            print("  branch inputs :", [i.name for i in attr.g.input])
            print("  branch outputs:", [o.name for o in attr.g.output])
            for subnode in attr.g.node:
                print("   ", subnode.op_type, subnode.name, "in=", list(subnode.input), "out=", list(subnode.output))


===== /backbone/network.0/conv1/If =====
inputs : ['/backbone/network.0/conv1/Equal_output_0']
outputs: ['/backbone/network.0/conv1/If_output_0']

  [then_branch]
  branch inputs : []
  branch outputs: ['/backbone/network.0/conv1/Squeeze_output_0']
    Squeeze /backbone/network.0/conv1/Squeeze in= ['/backbone/network.0/conv1/conv/Conv_output_0', '/backbone/network.0/conv1/Constant_3_output_0'] out= ['/backbone/network.0/conv1/Squeeze_output_0']

  [else_branch]
  branch inputs : []
  branch outputs: ['/backbone/network.0/conv1/Identity_output_0']
    Identity /backbone/network.0/conv1/Identity in= ['/backbone/network.0/conv1/conv/Conv_output_0'] out= ['/backbone/network.0/conv1/Identity_output_0']

===== /backbone/network.0/conv2/If =====
inputs : ['/backbone/network.0/conv2/Equal_output_0']
outputs: ['/backbone/network.0/conv2/If_output_0']

  [then_branch]
  branch inputs : []
  branch outputs: ['/backbone/network.0/conv2/Squeeze_output_0']
    Squeeze /backbone/network.0/conv2/Sque

In [1]:
!cd /home/slowman/Desktop/project/Thesis/finn/notebooks/onnx2bit && python pytorch2onnx.py

THIS_DIR      = /home/slowman/Desktop/project/Thesis/finn/notebooks/onnx2bit
NOTEBOOKS_DIR = /home/slowman/Desktop/project/Thesis/finn/notebooks
PROJECT_DIR   = /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn
CHECKPOINT : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/new_version/best_qtcn_model_last_step.pth
EXPORT_DIR : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports
Block 0: in_ch=1, out_ch=16, dilation=1
Block 1: in_ch=16, out_ch=16, dilation=2
Block 2: in_ch=16, out_ch=16, dilation=4
Checkpoint loaded.
Model config    : {'num_classes': 5, 'seq_len': 140, 'num_inputs': 1, 'num_channels': (16, 16, 16), 'kernel_size': 3, 'dropout': 0.1, 'causal': True, 'export_all_time_logits': False, 'expected_output_shape': '[N, 5, 1, 1]'}
Missing keys    : []
Unexpected keys : []
/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject

NameError: name 'out_a' is not defined

> /tmp/ipykernel_152/1163955116.py(1)<module>()
----> 1 diff = np.abs(out_a - out_b)
      2 print("max abs diff :", diff.max())
      3 print("mean abs diff:", diff.mean())
      4 print("argmax match :", (out_a.argmax(axis=1) == out_b.argmax(axis=1)).mean())
      5 print("out_a range  :", out_a.min(), out_a.max())



ipdb>  exit


In [10]:
from qonnx.core.modelwrapper import ModelWrapper

MODEL = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_convert_to_hw.onnx"
m = ModelWrapper(MODEL)

def is_hw(node):
    if node is None:
        return False
    if node.domain == "finn.custom_op.fpgadataflow":
        return True
    for a in node.attribute:
        if a.name == "backend":
            try:
                if a.s.decode("utf-8") == "fpgadataflow":
                    return True
            except Exception:
                pass
    return False

print("=== Add / Mul nodes between HW nodes ===")
for i, n in enumerate(m.graph.node):
    if n.op_type not in ["Add", "Mul", "Sub", "Div"]:
        continue

    preds = m.find_direct_predecessors(n) or []
    succs = m.find_direct_successors(n) or []

    pred_info = [(p.name, p.op_type, is_hw(p)) for p in preds]
    succ_info = [(s.name, s.op_type, is_hw(s)) for s in succs]

    hw_pred = any(is_hw(p) for p in preds)
    hw_succ = any(is_hw(s) for s in succs)

    if hw_pred or hw_succ:
        print(f"\n[{i}] {n.name} ({n.op_type})")
        print("  preds:", pred_info)
        print("  succs:", succ_info)

=== Add / Mul nodes between HW nodes ===

[65] Mul_6 (Mul)
  preds: [('MVAU_2', 'MVAU', True)]
  succs: [('Add_3', 'Add', False)]


In [2]:
%cd /home/slowman/Desktop/project/Thesis/finn/notebooks/onnx2bit
import warnings
warnings.filterwarnings("ignore")
%pdb on
%run onnx2bit.py

/home/slowman/Desktop/project/Thesis/finn/notebooks/onnx2bit
Automatic pdb calling has been turned ON
Building dataflow accelerator from /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_finn_streamlined.onnx
Intermediate outputs will be generated in /tmp/finn_dev_slowman
Final outputs will be generated in /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2
Build log is at /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/build_dataflow.log
Running step: step_streamline [1/25]
Running step: step_after_streamline_cleanup [2/25]
[PATCH] cleanup between step_streamline and step_convert_to_hw
[AbsorbScalarMulIntoThresholdPath] absorbed scalar scale=0.5 from Mul_0 into thresholds of MultiThreshold_3
[AbsorbScalarMulIntoThresholdPath] absorbed scalar scale=0.5 from Mul_1 into thresholds of MultiThreshold_6
[AbsorbScalarMulIntoThresholdPath] absorbed scalar scale=0.25 from Mul_2 int

%Warning-MODDUP: /tmp/finn_dev_slowman/vivado_stitch_proj_c4erxq81/finn_design_wrapper.v:7539:8: Duplicate declaration of module: 'fmpadding_axi'
 7539 | module fmpadding_axi #(
      |        ^~~~~~~~~~~~~
                 /tmp/finn_dev_slowman/vivado_stitch_proj_c4erxq81/finn_design_wrapper.v:471:8: ... Location of original declaration
  471 | module fmpadding_axi #(
      |        ^~~~~~~~~~~~~
                 ... For warning description see https://verilator.org/warn/MODDUP?v=4.224
                 ... Use "/* verilator lint_off MODDUP */" and lint_on around source to disable this message.
%Warning-MODDUP: /tmp/finn_dev_slowman/vivado_stitch_proj_c4erxq81/finn_design_wrapper.v:8201:8: Duplicate declaration of module: 'thresholding_axi'
 8201 | module thresholding_axi #(
      |        ^~~~~~~~~~~~~~~~
                 /tmp/finn_dev_slowman/vivado_stitch_proj_c4erxq81/finn_design_wrapper.v:89:8: ... Location of original declaration
   89 | module thresholding_axi #(
      |        

make: Entering directory '/tmp/finn_dev_slowman/verilator_fifosim_jei3sd43'
ccache g++  -I.  -MMD -I/usr/local/share/verilator/include -I/usr/local/share/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TRACE=0 -DVM_TRACE_FST=0 -DVM_TRACE_VCD=0 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-sign-compare -Wno-uninitialized -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable -Wno-shadow     --std=c++11  -DVL_THREADED -std=gnu++17 -Os -c -o verilator_fifosim.o verilator_fifosim.cpp
ccache g++  -I.  -MMD -I/usr/local/share/verilator/include -I/usr/local/share/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TRACE=0 -DVM_TRACE_FST=0 -DVM_TRACE_VCD=0 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-sign-compare -Wno-uninitialized -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable -Wno-shadow     --std=c++11  -DVL_THREADED -std=gnu++17 -Os -c -o verilated.o /usr/local/share/verilator/include/verilated.cpp
ccache g+

In [2]:
%cd /home/slowman/Desktop/project/Thesis/finn/notebooks/onnx2bit
import warnings
warnings.filterwarnings("ignore")
%pdb on
%run continue_build_from_patched_dataflow.py

/home/slowman/Desktop/project/Thesis/finn/notebooks/onnx2bit
Automatic pdb calling has been turned ON
Building dataflow accelerator from /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_target_fps_parallelization.onnx
Intermediate outputs will be generated in /tmp/finn_dev_slowman
Final outputs will be generated in /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2_force_final_hls
Build log is at /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2_force_final_hls/build_dataflow.log
Running step: step_force_final_mvau_hls [1/9]
> /home/slowman/Desktop/project/Thesis/finn/notebooks/onnx2bit/continue_build_from_patched_dataflow.py(51)step_force_final_mvau_hls()
     49 
     50     if target is None:
---> 51         raise RuntimeError("Could not find final MVAU with output shape [1, 1, 1, 5].")
     52 
     53     print("Forcing final MVAU to HLS:", target.name)



Traceback (most recent call last):
  File "/home/slowman/Desktop/project/Thesis/finn/src/finn/builder/build_dataflow.py", line 228, in build_dataflow_cfg
    model = transform_step(model, cfg)
  File "/home/slowman/Desktop/project/Thesis/finn/notebooks/onnx2bit/continue_build_from_patched_dataflow.py", line 51, in step_force_final_mvau_hls
    raise RuntimeError("Could not find final MVAU with output shape [1, 1, 1, 5].")
RuntimeError: Could not find final MVAU with output shape [1, 1, 1, 5].


ipdb>  exit


Build failed


In [11]:
from qonnx.core.modelwrapper import ModelWrapper

MODEL = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup.onnx"
m = ModelWrapper(MODEL)

def is_hw(node):
    if node is None:
        return False
    if node.domain == "finn.custom_op.fpgadataflow":
        return True
    for a in node.attribute:
        if a.name == "backend":
            try:
                if a.s.decode("utf-8") == "fpgadataflow":
                    return True
            except Exception:
                pass
    return False

print("MODEL =", MODEL)
print("\n=== Non-HW bridge / fork nodes in step_after_convert_to_hw_cleanup.onnx ===")

for i, n in enumerate(m.graph.node):
    preds = m.find_direct_predecessors(n) or []
    succs = m.find_consumers(n.output[0]) or []

    has_hw_pred = any(is_hw(p) for p in preds)
    has_hw_succ = any(is_hw(s) for s in succs)

    if (not is_hw(n)) and (has_hw_pred or has_hw_succ):
        print(f"\n[{i}] {n.name} ({n.op_type})")
        print("  preds:", [(p.name, p.op_type, is_hw(p)) for p in preds])
        print("  succs:", [(s.name, s.op_type, is_hw(s)) for s in succs])
        print("  inputs :", list(n.input))
        print("  outputs:", list(n.output))

MODEL = /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup.onnx

=== Non-HW bridge / fork nodes in step_after_convert_to_hw_cleanup.onnx ===

[1] MultiThreshold_0 (MultiThreshold)
  preds: [('Unsqueeze_0', 'Unsqueeze', False)]
  succs: [('DuplicateStreams_MultiThreshold_0', 'DuplicateStreams', True)]
  inputs : ['Unsqueeze_0_out0', 'MultiThreshold_0_param0']
  outputs: ['MultiThreshold_0_out0']

[3] Transpose_0 (Transpose)
  preds: [('DuplicateStreams_MultiThreshold_0', 'DuplicateStreams', True)]
  succs: [('MVAU_0', 'MVAU', True)]
  inputs : ['TEMKWh']
  outputs: ['Transpose_0_out0']

[4] Pad_0 (Pad)
  preds: [('DuplicateStreams_MultiThreshold_0', 'DuplicateStreams', True)]
  succs: [('Transpose_1', 'Transpose', False)]
  inputs : ['pmJjVV', 'Pad_0_param0', 'Pad_0_param1']
  outputs: ['Pad_0_out0']

[6] Transpose_1 (Transpose)
  preds: [('Pad_0', 'Pad', False)]
  succs: [('ConvolutionInputGenerator_0',

In [12]:
from qonnx.core.modelwrapper import ModelWrapper

MODEL = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup.onnx"
m = ModelWrapper(MODEL)

def is_hw(node):
    if node is None:
        return False
    if node.domain == "finn.custom_op.fpgadataflow":
        return True
    for a in node.attribute:
        if a.name == "backend":
            try:
                if a.s.decode("utf-8") == "fpgadataflow":
                    return True
            except Exception:
                pass
    return False

print("\n=== Fanout tensors touching HW nodes ===")
for n in m.graph.node:
    succs = m.find_consumers(n.output[0]) or []
    if len(succs) >= 2:
        hw_succs = [s for s in succs if is_hw(s)]
        if hw_succs:
            print(f"\nProducer: {n.name} ({n.op_type})")
            print("  output:", n.output[0])
            print("  succs :", [(s.name, s.op_type, is_hw(s)) for s in succs])


=== Fanout tensors touching HW nodes ===


In [13]:
from qonnx.core.modelwrapper import ModelWrapper

MODEL = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/pre_partition_cleanup.onnx"
m = ModelWrapper(MODEL)

for n in m.graph.node:
    if n.name in ["Mul_6", "Add_3"]:
        print("\nNODE:", n.name, n.op_type)
        print("  inputs :", list(n.input))
        print("  outputs:", list(n.output))
        for inp in n.input:
            init = m.get_initializer(inp)
            print("   ", inp, "is_initializer =", init is not None, "shape =", None if init is None else init.shape, "value =", init)
        preds = m.find_direct_predecessors(n) or []
        succs = m.find_direct_successors(n) or []
        print("  preds :", [(p.name, p.op_type) for p in preds if p is not None])
        print("  succs :", [(s.name, s.op_type) for s in succs if s is not None])


NODE: Mul_6 Mul
  inputs : ['MatMul_8_out0', 'Mul_6_param0']
  outputs: ['Mul_6_out0']
    MatMul_8_out0 is_initializer = False shape = None value = None
    Mul_6_param0 is_initializer = True shape = () value = 3.0517578e-05
  preds : [('MVAU_2', 'MVAU')]
  succs : [('Add_3', 'Add')]

NODE: Add_3 Add
  inputs : ['Mul_6_out0', 'Add_3_param0']
  outputs: ['global_out']
    Mul_6_out0 is_initializer = False shape = None value = None
    Add_3_param0 is_initializer = True shape = (5,) value = [-0.00427246 -0.00128174  0.00494385 -0.00323486  0.00054932]
  preds : [('Mul_6', 'Mul')]
  succs : []


In [14]:
import json
from pathlib import Path
import onnx
from onnx import helper
from qonnx.core.modelwrapper import ModelWrapper

IN_MODEL = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn_streamlined.onnx"
OUT_MODEL = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn_hw_out.onnx"
OUT_META = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_output_postprocess.json"

m = ModelWrapper(IN_MODEL)

# 找最后这两个节点
mul_node = next(n for n in m.graph.node if n.name == "Mul_6")
add_node = next(n for n in m.graph.node if n.name == "Add_3")

acc_tensor = mul_node.input[0]          # MatMul_8_out0
scale_name = mul_node.input[1]          # Mul_6_param0
bias_name = add_node.input[1]           # Add_3_param0

scale = m.get_initializer(scale_name)
bias = m.get_initializer(bias_name)

print("acc_tensor =", acc_tensor)
print("scale =", scale)
print("bias =", bias)

# 把 graph output 改成整数 accumulator 输出
# 先清空旧输出
while len(m.graph.output) > 0:
    m.graph.output.remove(m.graph.output[0])

acc_shape = m.get_tensor_shape(acc_tensor)
if acc_shape is None:
    acc_shape = [1, 5]

out_vi = helper.make_tensor_value_info(
    acc_tensor,
    onnx.TensorProto.FLOAT,   # 这里先用 FLOAT 占位，FINN datatype metadata 还在
    acc_shape
)
m.graph.output.append(out_vi)

# 可选：把最后两个无用节点删掉
# 这里只删 Add_3 和 Mul_6，前提是它们只服务于原 global_out
m.graph.node.remove(add_node)
m.graph.node.remove(mul_node)

m.save(OUT_MODEL)
print("Saved:", OUT_MODEL)

meta = {
    "acc_tensor": acc_tensor,
    "scale": float(scale.reshape(-1)[0]),
    "bias": [float(x) for x in bias.reshape(-1)],
}
with open(OUT_META, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved:", OUT_META)

StopIteration: 

> /tmp/ipykernel_145/1626204219.py(15)<module>()
     13 # 找最后这两个节点
     14 mul_node = next(n for n in m.graph.node if n.name == "Mul_6")
---> 15 add_node = next(n for n in m.graph.node if n.name == "Add_3")
     16 
     17 acc_tensor = mul_node.input[0]          # MatMul_8_out0



ipdb>  exit


In [21]:
from qonnx.core.modelwrapper import ModelWrapper

MODEL = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/pre_partition_cleanup.onnx"
m = ModelWrapper(MODEL)

def is_hw(node):
    if node is None:
        return False
    if getattr(node, "domain", "") == "finn.custom_op.fpgadataflow":
        return True
    for a in node.attribute:
        if a.name == "backend":
            try:
                if a.s.decode("utf-8") == "fpgadataflow":
                    return True
            except Exception:
                pass
    return False

print("=== Non-HW bridge / fork nodes in pre_partition_cleanup.onnx ===")
for i, n in enumerate(m.graph.node):
    if is_hw(n):
        continue

    preds = m.find_direct_predecessors(n) or []
    succs = m.find_direct_successors(n) or []

    hw_pred = any(is_hw(p) for p in preds if p is not None)
    hw_succ = any(is_hw(s) for s in succs if s is not None)
    is_fork = len(succs) > 1
    is_join = len(preds) > 1

    if hw_pred or hw_succ or is_fork or is_join:
        print(f"\n[{i}] {n.name} ({n.op_type})")
        print("  preds:", [(p.name, p.op_type, is_hw(p)) for p in preds if p is not None])
        print("  succs:", [(s.name, s.op_type, is_hw(s)) for s in succs if s is not None])
        print("  inputs:", list(n.input))
        print("  outputs:", list(n.output))
        for inp in n.input:
            init = m.get_initializer(inp)
            if init is not None:
                print("   init:", inp, "shape =", init.shape)

=== Non-HW bridge / fork nodes in pre_partition_cleanup.onnx ===

[1] MultiThreshold_0 (MultiThreshold)
  preds: [('Unsqueeze_0', 'Unsqueeze', False)]
  succs: [('DuplicateStreams_MultiThreshold_0', 'DuplicateStreams', True)]
  inputs: ['Unsqueeze_0_out0', 'MultiThreshold_0_param0']
  outputs: ['MultiThreshold_0_out0']
   init: MultiThreshold_0_param0 shape = (1, 255)

[3] Mul_0 (Mul)
  preds: [('DuplicateStreams_MultiThreshold_0', 'DuplicateStreams', True)]
  succs: [('Pad_0', 'Pad', False)]
  inputs: ['jmOeMq', 'Mul_0_param0']
  outputs: ['Mul_0_out0']
   init: Mul_0_param0 shape = ()

[4] Transpose_0 (Transpose)
  preds: [('DuplicateStreams_MultiThreshold_0', 'DuplicateStreams', True)]
  succs: [('MVAU_0', 'MVAU', True)]
  inputs: ['AcBoPA']
  outputs: ['Transpose_0_out0']

[17] MultiThreshold_2 (MultiThreshold)
  preds: [('MatMul_1', 'MatMul', False)]
  succs: [('AddStreams_Add_0', 'AddStreams', True)]
  inputs: ['MatMul_2_out0', 'MultiThreshold_3_param0']
  outputs: ['MultiThresho

In [11]:
from qonnx.core.modelwrapper import ModelWrapper

model = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_debug/intermediate_models/step_convert_to_hw.onnx")
graph = model.graph
nodes = list(graph.node)

producer = {}
consumers = {}
for n in nodes:
    for o in n.output:
        producer[o] = n
    for i in n.input:
        consumers.setdefault(i, []).append(n)

def is_hw_node(n):
    return n.domain == "finn.custom_op.fpgadataflow"

bad_nodes = []

for n in nodes:
    prev_hw = any(producer.get(i, None) is not None and is_hw_node(producer[i]) for i in n.input)
    next_hw = any(is_hw_node(c) for o in n.output for c in consumers.get(o, []))

    if (not is_hw_node(n)) and prev_hw and next_hw:
        bad_nodes.append((n.name, n.op_type, n.domain))

print("Potential partition breakers:")
for x in bad_nodes:
    print(x)

Potential partition breakers:


In [18]:
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from finn.transformation.streamline import Streamline

model = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_qtcn_finn.onnx")

# 先做一遍 shape inference
model = model.transform(InferShapes())

# 先不要整个 build，手动做到 streamline 前后
model = model.transform(Streamline())

# 找所有 Conv
convs = model.get_nodes_by_op_type("Conv")
print("num convs =", len(convs))

for i, node in enumerate(convs):
    print(f"\n=== Conv {i} ===")
    print("name:", node.name)
    print("inputs:", node.input)
    print("outputs:", node.output)
    print("input shape:", model.get_tensor_shape(node.input[0]))
    print("output shape:", model.get_tensor_shape(node.output[0]))
    w = model.get_initializer(node.input[1])
    print("weight shape:", None if w is None else w.shape)

# 找所有带 0 shape 的张量
# for node in model.graph.node:
#     bad = False
#     outs = []
#     for o in node.output:
#         shp = model.get_tensor_shape(o)
#         outs.append((o, shp))
#         if shp is not None and 0 in shp:
#             bad = True
#     if bad:
#         print(f"{node.op_type:15s} {node.name}")
#         print("  inputs :", [(i, model.get_tensor_shape(i)) for i in node.input if i != ""])
#         print("  outputs:", outs)
#         print()

# 最后单独试 LowerConvsToMatMul
model2 = model.transform(LowerConvsToMatMul())

KeyboardInterrupt: 

> /home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/core/modelwrapper.py(317)rename_tensor()
    315         # sweep over node i/o
    316         for n in graph.node:
--> 317             if old_name in n.input:
    318                 n.input[list(n.input).index(old_name)] = new_name
    319             if old_name in n.output:



ipdb>  exit


In [11]:
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.infer_shapes import InferShapes

model = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_qtcn_finn.onnx")
model = model.transform(InferShapes())

found = False
for node in model.graph.node:
    for o in node.output:
        shp = model.get_tensor_shape(o)
        if shp is not None and 0 in shp:
            print("FIRST BAD NODE:")
            print("  op   :", node.op_type)
            print("  name :", node.name)
            print("  inputs:")
            for i in node.input:
                if i != "":
                    print("   ", i, model.get_tensor_shape(i))
            print("  outputs:")
            for oo in node.output:
                print("   ", oo, model.get_tensor_shape(oo))
            found = True
            break
    if found:
        break

FIRST BAD NODE:
  op   : MultiThreshold
  name : 
  inputs:
    /backbone/network.0/conv1/If_output_0 []
    l7r56E [0, 255]
  outputs:
    8ZlWUC [0, 0, 0]


In [ ]:
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.infer_shapes import InferShapes

model = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_qtcn_finn.onnx")
model = model.transform(InferShapes())

ifs = [n for n in model.graph.node if n.op_type == "If"]
print("num If =", len(ifs))
for n in ifs[:20]:
    print(n.name, n.input, n.output)

In [22]:
print("num If =", len([n for n in model.graph.node if n.op_type == "If"]))
for n in model.get_nodes_by_op_type("Conv"):
    print(n.name, model.get_tensor_shape(n.input[0]), model.get_tensor_shape(n.output[0]))

num If = 9
/backbone/network.0/downsample/conv/Conv [1, 1, 1, 140] [1, 16, 1, 140]
/backbone/network.0/conv1/conv/Conv [1, 1, 1, 142] [1, 16, 1, 140]
/backbone/network.0/conv2/conv/Conv [1, 16, 1, 0] [1, 16, 1, 0]
/backbone/network.1/conv1/conv/Conv [1, 16, 1, 0] [1, 16, 1, 0]
/backbone/network.1/conv2/conv/Conv [1, 16, 1, 0] [1, 16, 1, 0]
/backbone/network.2/downsample/conv/Conv [0, 0, 1, 0] [0, 32, 1, 0]
/backbone/network.2/conv1/conv/Conv [1, 16, 1, 0] [1, 32, 1, 0]
/backbone/network.2/conv2/conv/Conv [1, 32, 1, 0] [1, 32, 1, 0]


In [25]:
from qonnx.core.modelwrapper import ModelWrapper
from collections import Counter

fn = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_convert_to_hw.onnx"
m = ModelWrapper(fn)

print("=== inputs ===")
for x in m.graph.input:
    print(x.name, m.get_tensor_shape(x.name))

print("\n=== outputs ===")
for x in m.graph.output:
    print(x.name, m.get_tensor_shape(x.name))

print("\n=== op histogram ===")
ops = Counter([n.op_type for n in m.graph.node])
for k, v in sorted(ops.items()):
    print(f"{k}: {v}")

print("\n=== nodes ===")
for i, n in enumerate(m.graph.node):
    print(i, n.name, n.op_type, n.domain)

=== inputs ===
global_in [1, 1, 140]

=== outputs ===
global_out [0, 5]

=== op histogram ===
Add: 6
AveragePool: 1
Concat: 18
ConstantOfShape: 12
ConvolutionInputGenerator: 6
Equal: 14
Expand: 12
Gather: 8
Identity: 7
If: 2
MVAU: 9
Mul: 25
MultiThreshold: 13
Reshape: 12
Shape: 20
Thresholding: 7
Transpose: 16
Unsqueeze: 13
Where: 12

=== nodes ===
0 Identity_0 Identity 
1 Identity_1 Identity 
2 Identity_2 Identity 
3 Identity_3 Identity 
4 Identity_4 Identity 
5 Identity_5 Identity 
6 Identity_6 Identity 
7 MultiThreshold_0 MultiThreshold qonnx.custom_op.general
8 Thresholding_0 Thresholding finn.custom_op.fpgadataflow
9 Mul_0 Mul 
10 Unsqueeze_0 Unsqueeze 
11 Shape_0 Shape 
12 MultiThreshold_1 MultiThreshold qonnx.custom_op.general
13 Gather_0 Gather 
14 Transpose_0 Transpose 
15 MVAU_0 MVAU finn.custom_op.fpgadataflow
16 Unsqueeze_1 Unsqueeze 
17 Unsqueeze_2 Unsqueeze 
18 Transpose_1 Transpose 
19 Concat_0 Concat 
20 Concat_1 Concat 
21 Mul_1 Mul 
22 Reshape_0 Reshape 
23 Reshape_1 

In [26]:
from qonnx.core.modelwrapper import ModelWrapper

fn = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_convert_to_hw.onnx"
m = ModelWrapper(fn)

suspect = {
    "Transpose", "Reshape", "Flatten", "Pad", "Slice",
    "Concat", "Squeeze", "Unsqueeze", "Identity"
}

print("=== suspicious nodes ===")
for i, n in enumerate(m.graph.node):
    if n.op_type in suspect:
        print(i, n.name, n.op_type, n.input, "->", n.output)

=== suspicious nodes ===
0 Identity_0 Identity ['Expand_10_param0'] -> ['Identity_0_out0']
1 Identity_1 Identity ['Expand_8_param0'] -> ['Identity_1_out0']
2 Identity_2 Identity ['Expand_4_param0'] -> ['Identity_2_out0']
3 Identity_3 Identity ['Expand_4_param0'] -> ['Identity_3_out0']
4 Identity_4 Identity ['Expand_4_param0'] -> ['Identity_4_out0']
5 Identity_5 Identity ['Expand_2_param0'] -> ['Identity_5_out0']
6 Identity_6 Identity ['Expand_0_param0'] -> ['Identity_6_out0']
10 Unsqueeze_0 Unsqueeze ['Mul_0_out0', 'Unsqueeze_0_param0'] -> ['Unsqueeze_0_out0']
14 Transpose_0 Transpose ['MultiThreshold_2_out0'] -> ['Transpose_0_out0']
16 Unsqueeze_1 Unsqueeze ['Gather_0_out0', 'Unsqueeze_1_param0'] -> ['Unsqueeze_1_out0']
17 Unsqueeze_2 Unsqueeze ['Gather_0_out0', 'Unsqueeze_2_param0'] -> ['Unsqueeze_2_out0']
18 Transpose_1 Transpose ['MatMul_0_out0'] -> ['Transpose_1_out0']
19 Concat_0 Concat ['Unsqueeze_1_out0', 'Concat_0_param0', 'Concat_0_param1', 'Concat_0_param2'] -> ['Concat_0_ou

In [1]:
from qonnx.core.modelwrapper import ModelWrapper

MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_qonnx_clean.onnx"

model = ModelWrapper(MODEL_PATH)

graph_input = model.graph.input[0].name
print("graph input name:", graph_input)

# 找所有直接吃 graph input 的节点
first_nodes = [n for n in model.graph.node if graph_input in n.input]

print("\nNodes directly connected to graph input:")
for i, n in enumerate(first_nodes):
    print(f"[{i}] name={n.name!r}, op_type={n.op_type}, domain={n.domain}")
    print("    inputs :", list(n.input))
    print("    outputs:", list(n.output))

# 找第一个 Quant 节点
quant_node = None
for n in first_nodes:
    if n.op_type == "Quant":
        quant_node = n
        break

if quant_node is None:
    print("\nNo direct Quant node found after graph input.")
    print("This can happen if quantization got folded/rewritten.")
else:
    print("\nFound input Quant node:")
    print("name   :", quant_node.name)
    print("domain :", quant_node.domain)
    print("inputs :", list(quant_node.input))
    print("outputs:", list(quant_node.output))

    scale_name = quant_node.input[1]
    zeropt_name = quant_node.input[2]
    bw_name = quant_node.input[3]

    scale = model.get_initializer(scale_name)
    zeropt = model.get_initializer(zeropt_name)
    bitwidth = model.get_initializer(bw_name)

    print("\nQuant parameters:")
    print("scale name    :", scale_name)
    print("scale value   :", scale)
    print("zeropt name   :", zeropt_name)
    print("zeropt value  :", zeropt)
    print("bitwidth name :", bw_name)
    print("bitwidth value:", bitwidth)

    attrs = {a.name: ModelWrapper._parse_attribute_proto(a) for a in quant_node.attribute}
    print("\nQuant attributes:")
    for k, v in attrs.items():
        print(f"{k}: {v}")

graph input name: x.785

Nodes directly connected to graph input:
[0] name='/quant_in/act_quant/export_handler/Quant', op_type=Quant, domain=onnx.brevitas
    inputs : ['x.785', '/quant_in/act_quant/export_handler/Constant_1_output_0', '/quant_in/act_quant/export_handler/Constant_2_output_0', '/quant_in/act_quant/export_handler/Constant_output_0']
    outputs: ['/quant_in/act_quant/export_handler/Quant_output_0']

Found input Quant node:
name   : /quant_in/act_quant/export_handler/Quant
domain : onnx.brevitas
inputs : ['x.785', '/quant_in/act_quant/export_handler/Constant_1_output_0', '/quant_in/act_quant/export_handler/Constant_2_output_0', '/quant_in/act_quant/export_handler/Constant_output_0']
outputs: ['/quant_in/act_quant/export_handler/Quant_output_0']

Quant parameters:
scale name    : /quant_in/act_quant/export_handler/Constant_1_output_0
scale value   : 0.041415304
zeropt name   : /quant_in/act_quant/export_handler/Constant_2_output_0
zeropt value  : 0.0
bitwidth name : /quant

AttributeError: type object 'ModelWrapper' has no attribute '_parse_attribute_proto'

In [3]:
from qonnx.core.modelwrapper import ModelWrapper

m = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_streamline.onnx")
iname = m.graph.input[0].name
print("input name:", iname)
print("input shape:", m.get_tensor_shape(iname))
print("input dtype:", m.get_tensor_datatype(iname))

input name: global_in
input shape: [1, 1, 140]
input dtype: FLOAT32


In [5]:
!cd /home/slowman/Desktop/project/Thesis/finn/notebooks/onnx2bit && python outputcheck.py

Traceback (most recent call last):
  File "/home/slowman/Desktop/project/Thesis/finn/notebooks/onnx2bit/outputcheck.py", line 1, in <module>
    from new_version.newnet import ECG5000FullQuantTCN
ModuleNotFoundError: No module named 'new_version'
